In [19]:
import cv2
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cpu


In [21]:
class_names = {
    0: "surprise",
    1: "fear",
    2: "disgust",
    3: "happy",
    4: "sad",
    5: "angry",
    6: "neutral"
}

In [22]:
model = models.resnet34(pretrained=False)

model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, 7)
)

model.load_state_dict(torch.load("emotion_model_rafdbv1.pth", map_location=device))

model = model.to(device)
model.eval()

print("Model loaded successfully")

Model loaded successfully


In [23]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [24]:
# Load OpenCV's built-in face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Webcam not found")
else:
    print("Webcam started. Press 'q' to quit.")

while True:
    ret, frame = cap.read()

    if not ret:
        print("Failed to read frame")
        break

    # Convert to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(80, 80)
    )

    # If at least one face is detected
    if len(faces) > 0:
        # Take the largest face
        largest_face = max(faces, key=lambda box: box[2] * box[3])
        x, y, w, h = largest_face

        # Add small padding around face
        padding_x = int(w * 0.10)
        padding_y = int(h * 0.05)

        x1 = max(x + padding_x, 0)
        y1 = max(y + padding_y, 0)
        x2 = min(x + w - padding_x, frame.shape[1])
        y2 = min(y + h - padding_y, frame.shape[0])

        face_crop = frame[y1:y2, x1:x2]

        # Convert BGR face crop to RGB
        rgb_face = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        pil_face = Image.fromarray(rgb_face)

        # Preprocess face
        input_tensor = transform(pil_face).unsqueeze(0)
        input_tensor = input_tensor.to(device)

        # Predict emotion
        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = torch.softmax(outputs, dim=1)
            confidence, predicted_class = torch.max(probabilities, 1)

        emotion = class_names[predicted_class.item()]
        confidence_score = confidence.item()

        text = f"{emotion} ({confidence_score:.2f})"

        # Draw face box
        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        # Draw prediction text
        cv2.putText(
            frame,
            text,
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2
        )

    else:
        cv2.putText(
            frame,
            "No face detected",
            (30, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 0, 255),
            2
        )

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Webcam started. Press 'q' to quit.
